In [1]:
import os, json, random, time
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForConditionalGeneration
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import nltk
import torch
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
 
# GPU performance settings
torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
print("=" * 50)
print(f"  PyTorch   : {torch.__version__}")
print(f"  CUDA      : {torch.version.cuda}")
print(f"  Device    : {DEVICE}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"  GPU       : {gpu.name}")
    print(f"  VRAM      : {gpu.total_memory / 1e9:.1f} GB")
    print(f"  Compute   : sm_{gpu.major}{gpu.minor}")
print("=" * 50)

  PyTorch   : 2.6.0+cu124
  CUDA      : 12.4
  Device    : cuda
  GPU       : Tesla P100-PCIE-16GB
  VRAM      : 17.1 GB
  Compute   : sm_60


In [ ]:

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # fixes memory fragmentation

TRAIN_IMAGES = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017"
VAL_IMAGES   = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017"
ANN_DIR      = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations"
TRAIN_ANN    = os.path.join(ANN_DIR, "captions_train2017.json")
VAL_ANN      = os.path.join(ANN_DIR, "captions_val2017.json")

CONFIG = {
    "model_name"    : "Salesforce/blip-image-captioning-base",
    "batch_size"    : 8,       
    "epochs"        : 5,
    "lr"            : 5e-5,
    "weight_decay"  : 0.01,
    "max_grad_norm" : 1.0,
    "max_length"    : 64,
    "train_samples" : 80000,
    "val_samples"   : 5000,
    "num_workers"   : 4,
    "save_dir"      : "/kaggle/working/blip_coco",
    "seed"          : 42,
    "log_every"     : 100,
    "device"        : torch.device("cuda"),
    "grad_accum" : 1
}

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
random.seed(CONFIG["seed"])
os.makedirs(CONFIG["save_dir"], exist_ok=True)

torch.cuda.empty_cache()
print(f"Free VRAM : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")
print(f"Batch size : {CONFIG['batch_size']}")

Free VRAM : 9.6 GB
Batch size : 8


In [ ]:
class COCOCaptionDataset(Dataset):
    def __init__(self, image_dir, ann_file, processor, max_samples=None):
        # Load annotation JSON
        with open(ann_file, 'r') as f:
            data = json.load(f)

        self.id_to_file = {
            img['id']: img['file_name']
            for img in data['images']
        }
        self.annotations = data['annotations']

        if max_samples:
            self.annotations = self.annotations[:max_samples]

        self.image_dir    = image_dir
        self.processor    = processor
        self.pad_token_id = processor.tokenizer.pad_token_id

        print(f"  Loaded : {len(self.annotations):,} samples")
        print(f"  Images : {image_dir}")

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann     = self.annotations[idx]
        caption = ann['caption'].strip()

        img_file = self.id_to_file[ann['image_id']]
        img_path = os.path.join(self.image_dir, img_file)
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (384, 384))

        enc = self.processor(
            images=image,
            text=caption,
            padding='max_length',
            truncation=True,
            max_length=CONFIG['max_length'],
            return_tensors='pt'
        )

        input_ids = enc['input_ids'].squeeze()

        # Labels: padding positions → -100 so loss ignores them
        labels = input_ids.clone()
        labels[labels == self.pad_token_id] = -100

        return {
            'pixel_values'  : enc['pixel_values'].squeeze(),
            'input_ids'     : input_ids,
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels'        : labels
        }

In [ ]:
print("Loading BLIP processor...")
processor = BlipProcessor.from_pretrained(CONFIG["model_name"])

print("Loading BLIP model (pre-trained on 129M pairs)...")
model = BlipForConditionalGeneration.from_pretrained(CONFIG["model_name"])
model = model.to(CONFIG["device"])

total     = sum(p.numel() for p in model.parameters()) / 1e6
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Total params     : {total:.1f}M")
print(f"Trainable params : {trainable:.1f}M")

# ── Datasets ─────────────────────────────────────────────────
print("\nPreparing datasets...")
print("Train:")
train_dataset = COCOCaptionDataset(TRAIN_IMAGES, TRAIN_ANN, processor, CONFIG["train_samples"])
print("Validation:")
val_dataset   = COCOCaptionDataset(VAL_IMAGES,   VAL_ANN,   processor, CONFIG["val_samples"])


train_loader = DataLoader(
    train_dataset,
    batch_size      = CONFIG["batch_size"],
    shuffle         = True,
    num_workers     = CONFIG["num_workers"],
    pin_memory      = True,
    prefetch_factor = 2,
    persistent_workers = True,
    drop_last       = True
)

val_loader = DataLoader(
    val_dataset,
    batch_size      = CONFIG["batch_size"],
    shuffle         = False,
    num_workers     = CONFIG["num_workers"],
    pin_memory      = True,
    prefetch_factor = 2,
    persistent_workers = True
)

print(f"\nTrain batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(f"Steps/epoch   : {len(train_loader):,}")

Loading BLIP processor...
Loading BLIP model (pre-trained on 129M pairs)...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Total params     : 224.0M
Trainable params : 224.0M

Preparing datasets...
Train:
  Loaded : 80,000 samples
  Images : /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017
Validation:
  Loaded : 5,000 samples
  Images : /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017

Train batches : 10,000
Val batches   : 625
Steps/epoch   : 10,000


In [ ]:
def evaluate_bleu(model, processor, loader, device, num_batches=60):
    
    model.eval()
    references  = []    # ground truth captions
    hypotheses  = []    # model-generated captions

    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= num_batches:
                break

            # Move images to GPU (non_blocking=True → async transfer)
            pixel_values = batch['pixel_values'].to(device, non_blocking=True)

            # Generate captions (beam search for quality)
            generated_ids = model.generate(
                pixel_values  = pixel_values,
                max_new_tokens = 50,
                num_beams      = 4,
                early_stopping = True
            )

            preds = processor.batch_decode(generated_ids, skip_special_tokens=True)

            gt_ids = batch['input_ids'].clone()
            gt_ids[gt_ids == -100] = processor.tokenizer.pad_token_id
            gts   = processor.batch_decode(gt_ids, skip_special_tokens=True)

            for pred, gt in zip(preds, gts):
                hypotheses.append(pred.lower().split())
                references.append([gt.lower().split()])

    smoother = SmoothingFunction().method1
    bleu4    = corpus_bleu(references, hypotheses, smoothing_function=smoother)
    model.train()
    return round(bleu4 * 100, 2)


In [28]:
from torch.cuda.amp import GradScaler, autocast
optimizer = AdamW(
    model.parameters(),
    lr           = CONFIG["lr"],
    weight_decay = CONFIG["weight_decay"]
)

scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])

scaler = GradScaler()

total_steps = len(train_loader) * CONFIG["epochs"]
print(f"Optimizer   : AdamW  (lr={CONFIG['lr']}, wd={CONFIG['weight_decay']})")
print(f"Scheduler   : CosineAnnealingLR  (T_max={CONFIG['epochs']})")
print(f"Precision   : fp16 (GradScaler enabled)")
print(f"Total steps : {total_steps:,}")


Optimizer   : AdamW  (lr=5e-05, wd=0.01)
Scheduler   : CosineAnnealingLR  (T_max=5)
Precision   : fp16 (GradScaler enabled)
Total steps : 50,000


/tmp/ipykernel_988/3580230136.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
best_bleu = 0.0
history   = []

print("\n" + "="*55)
print("  TRAINING START")
print(f"  Epochs : {CONFIG['epochs']}  |  Batch : {CONFIG['batch_size']}  |  fp16 : ON")
print("="*55 + "\n")

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    epoch_loss = 0.0
    epoch_start = time.time()
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):

        # ── Move data to GPU (non_blocking = async, faster) ──
        pixel_values   = batch['pixel_values'].to(DEVICE, non_blocking=True)
        input_ids      = batch['input_ids'].to(DEVICE, non_blocking=True)
        attention_mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
        labels         = batch['labels'].to(DEVICE, non_blocking=True)

        # ── Forward pass — fixed autocast syntax ─────────────
        with torch.amp.autocast('cuda'):        # replaces deprecated torch.cuda.amp.autocast()
            outputs = model(
                pixel_values   = pixel_values,
                input_ids      = input_ids,
                attention_mask = attention_mask,
                labels         = labels
            )
            loss = outputs.loss / CONFIG["grad_accum"]

        # ── Backward pass ─────────────────────────────────────
        scaler.scale(loss).backward()

        # ── Optimizer step (every grad_accum steps) ───────────
        if (step + 1) % CONFIG["grad_accum"] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        epoch_loss += loss.item() * CONFIG["grad_accum"]

        # ── Log progress ──────────────────────────────────────
        if (step + 1) % CONFIG["log_every"] == 0:
            avg_loss = epoch_loss / (step + 1)
            elapsed  = time.time() - epoch_start
            eta_sec  = (elapsed / (step + 1)) * (len(train_loader) - step - 1)
            eta_min  = int(eta_sec // 60)
            print(f"  Ep {epoch} | Step {step+1:>4}/{len(train_loader)} "
                  f"| Loss: {avg_loss:.4f} | ETA: {eta_min}m")

    # ── End of epoch ──────────────────────────────────────────
    scheduler.step()
    avg_loss     = epoch_loss / len(train_loader)
    epoch_time   = int((time.time() - epoch_start) / 60)

    # ── Evaluate BLEU-4 ───────────────────────────────────────
    print(f"\n  Evaluating BLEU-4 (epoch {epoch})...")
    bleu4 = evaluate_bleu(model, processor, val_loader, DEVICE)

    # ── Save checkpoint ───────────────────────────────────────
    ckpt_path = os.path.join(CONFIG["save_dir"], f"epoch_{epoch}")
    model.save_pretrained(ckpt_path)
    processor.save_pretrained(ckpt_path)

    if bleu4 > best_bleu:
        best_bleu = bleu4
        best_path = os.path.join(CONFIG["save_dir"], "best_model")
        model.save_pretrained(best_path)
        processor.save_pretrained(best_path)
        best_marker = "  ✅ NEW BEST"
    else:
        best_marker = ""

    




  TRAINING START
  Epochs : 5  |  Batch : 8  |  fp16 : ON

  Ep 1 | Step  100/10000 | Loss: 2.6199 | ETA: 110m
  Ep 1 | Step  200/10000 | Loss: 2.3906 | ETA: 108m
  Ep 1 | Step  300/10000 | Loss: 2.3045 | ETA: 106m
  Ep 1 | Step  400/10000 | Loss: 2.2704 | ETA: 105m
  Ep 1 | Step  500/10000 | Loss: 2.2477 | ETA: 104m
  Ep 1 | Step  600/10000 | Loss: 2.2319 | ETA: 103m
  Ep 1 | Step  700/10000 | Loss: 2.2214 | ETA: 102m
  Ep 1 | Step  800/10000 | Loss: 2.2091 | ETA: 101m
  Ep 1 | Step  900/10000 | Loss: 2.2055 | ETA: 100m
  Ep 1 | Step 1000/10000 | Loss: 2.1976 | ETA: 98m
  Ep 1 | Step 1100/10000 | Loss: 2.1868 | ETA: 97m
  Ep 1 | Step 1200/10000 | Loss: 2.1828 | ETA: 96m
  Ep 1 | Step 1300/10000 | Loss: 2.1796 | ETA: 95m
  Ep 1 | Step 1400/10000 | Loss: 2.1748 | ETA: 94m
  Ep 1 | Step 1500/10000 | Loss: 2.1668 | ETA: 93m
  Ep 1 | Step 1600/10000 | Loss: 2.1629 | ETA: 92m
  Ep 1 | Step 1700/10000 | Loss: 2.1584 | ETA: 91m
  Ep 1 | Step 1800/10000 | Loss: 2.1550 | ETA: 90m
  Ep 1 | Step

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ──────────────────────────────────────────────────
  Epoch    : 1/5
  Loss     : 2.0413
  BLEU-4   : 7.29    ✅ NEW BEST
  Time     : 109m
  Saved    : /kaggle/working/blip_coco/epoch_1
  ──────────────────────────────────────────────────

  Ep 2 | Step  100/10000 | Loss: 1.6405 | ETA: 109m
  Ep 2 | Step  200/10000 | Loss: 1.6419 | ETA: 107m
  Ep 2 | Step  300/10000 | Loss: 1.6509 | ETA: 106m
  Ep 2 | Step  400/10000 | Loss: 1.6507 | ETA: 105m
  Ep 2 | Step  500/10000 | Loss: 1.6510 | ETA: 104m
  Ep 2 | Step  600/10000 | Loss: 1.6553 | ETA: 103m
  Ep 2 | Step  700/10000 | Loss: 1.6551 | ETA: 102m
  Ep 2 | Step  800/10000 | Loss: 1.6543 | ETA: 101m
  Ep 2 | Step  900/10000 | Loss: 1.6540 | ETA: 99m
  Ep 2 | Step 1000/10000 | Loss: 1.6560 | ETA: 98m
  Ep 2 | Step 1100/10000 | Loss: 1.6571 | ETA: 97m
  Ep 2 | Step 1200/10000 | Loss: 1.6583 | ETA: 96m
  Ep 2 | Step 1300/10000 | Loss: 1.6581 | ETA: 95m
  Ep 2 | Step 1400/10000 | Loss: 1.6605 | ETA: 94m
  Ep 2 | Step 1500/10000 | Loss: 1.6

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


  ──────────────────────────────────────────────────
  Epoch    : 3/5
  Loss     : 1.3720
  BLEU-4   : 8.03  
  Time     : 109m
  Saved    : /kaggle/working/blip_coco/epoch_3
  ──────────────────────────────────────────────────

  Ep 4 | Step  100/10000 | Loss: 1.0486 | ETA: 109m
  Ep 4 | Step  200/10000 | Loss: 1.0353 | ETA: 107m
  Ep 4 | Step  300/10000 | Loss: 1.0272 | ETA: 106m
  Ep 4 | Step  400/10000 | Loss: 1.0251 | ETA: 105m
  Ep 4 | Step  500/10000 | Loss: 1.0296 | ETA: 104m
  Ep 4 | Step  600/10000 | Loss: 1.0301 | ETA: 103m
  Ep 4 | Step  700/10000 | Loss: 1.0253 | ETA: 102m
  Ep 4 | Step  800/10000 | Loss: 1.0214 | ETA: 101m
  Ep 4 | Step  900/10000 | Loss: 1.0186 | ETA: 99m
  Ep 4 | Step 1000/10000 | Loss: 1.0148 | ETA: 98m
  Ep 4 | Step 1100/10000 | Loss: 1.0159 | ETA: 97m
  Ep 4 | Step 1200/10000 | Loss: 1.0155 | ETA: 96m
  Ep 4 | Step 1300/10000 | Loss: 1.0142 | ETA: 95m
  Ep 4 | Step 1400/10000 | Loss: 1.0134 | ETA: 94m
  Ep 4 | Step 1500/10000 | Loss: 1.0118 | ETA: 9

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ──────────────────────────────────────────────────
  Epoch    : 4/5
  Loss     : 1.0174
  BLEU-4   : 8.73  
  Time     : 109m
  Saved    : /kaggle/working/blip_coco/epoch_4
  ──────────────────────────────────────────────────

  Ep 5 | Step  100/10000 | Loss: 0.7450 | ETA: 109m
  Ep 5 | Step  200/10000 | Loss: 0.7441 | ETA: 107m
  Ep 5 | Step  300/10000 | Loss: 0.7340 | ETA: 106m
  Ep 5 | Step  400/10000 | Loss: 0.7297 | ETA: 105m
  Ep 5 | Step  500/10000 | Loss: 0.7290 | ETA: 104m
  Ep 5 | Step  600/10000 | Loss: 0.7295 | ETA: 103m
  Ep 5 | Step  700/10000 | Loss: 0.7273 | ETA: 102m
  Ep 5 | Step  800/10000 | Loss: 0.7272 | ETA: 101m
  Ep 5 | Step  900/10000 | Loss: 0.7278 | ETA: 99m
  Ep 5 | Step 1000/10000 | Loss: 0.7255 | ETA: 98m
  Ep 5 | Step 1100/10000 | Loss: 0.7242 | ETA: 97m
  Ep 5 | Step 1200/10000 | Loss: 0.7239 | ETA: 96m
  Ep 5 | Step 1300/10000 | Loss: 0.7240 | ETA: 95m
  Ep 5 | Step 1400/10000 | Loss: 0.7232 | ETA: 94m
  Ep 5 | Step 1500/10000 | Loss: 0.7215 | ETA: 9

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ──────────────────────────────────────────────────
  Epoch    : 5/5
  Loss     : 0.7123
  BLEU-4   : 9.15  
  Time     : 109m
  Saved    : /kaggle/working/blip_coco/epoch_5
  ──────────────────────────────────────────────────

  TRAINING COMPLETE
  Best BLEU-4 : 9.2

Epoch History:
  Epoch    Loss       BLEU-4
  ──────────────────────────────
  1        2.0413     7.29
  2        1.6911     9.2
  3        1.372      8.03
  4        1.0174     8.73
  5        0.7123     9.15


In [30]:
def generate_caption(image_path, model, processor, device):
    """Generate a caption for one image using trained model."""
    model.eval()
    image  = Image.open(image_path).convert('RGB')
    inputs = processor(images=image, return_tensors='pt').to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens = 50,
            num_beams      = 4,
            early_stopping = True
        )

    return processor.decode(output[0], skip_special_tokens=True)


# ── Load best saved model ─────────────────────────────────────
print("Loading best model...")
best_path      = os.path.join(CONFIG["save_dir"], "best_model")
best_processor = BlipProcessor.from_pretrained(best_path)
best_model     = BlipForConditionalGeneration.from_pretrained(best_path).to(DEVICE)

# ── Load validation annotations for ground truth ─────────────
with open(VAL_ANN, 'r') as f:
    val_data = json.load(f)

# Build image_id → captions lookup
id_to_captions = {}
for ann in val_data['annotations']:
    id_to_captions.setdefault(ann['image_id'], []).append(ann['caption'])

# ── Test on 5 random validation images ───────────────────────
sample_images = random.sample(val_data['images'], 5)

print("\n" + "="*55)
print("  SAMPLE CAPTIONS FROM TRAINED MODEL")
print("="*55)

for img_info in sample_images:
    img_path  = os.path.join(VAL_IMAGES, img_info['file_name'])
    predicted = generate_caption(img_path, best_model, best_processor, DEVICE)
    ground_ts = id_to_captions.get(img_info['id'], ["N/A"])

    print(f"\n  Image      : {img_info['file_name']}")
    print(f"  Predicted  : {predicted}")
    print(f"  Ground truth (1 of 5): {ground_ts[0]}")

print("\n" + "="*55)

Loading best model...


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]


  SAMPLE CAPTIONS FROM TRAINED MODEL

  Image      : 000000301061.jpg
  Predicted  : a person standing next to a truck in a building.
  Ground truth (1 of 5): A person is moving green hay towards an elephant that is inside the back of a white truck.

  Image      : 000000261982.jpg
  Predicted  : a skateboarder does a trick on his skateboard.
  Ground truth (1 of 5): A man flying through the air while riding a skateboard.

  Image      : 000000273760.jpg
  Predicted  : a man holding a tennis racquet while standing on a court.
  Ground truth (1 of 5): A man standing on top of a tennis court holding a racquet.

  Image      : 000000248400.jpg
  Predicted  : a man standing in front of a pizza box.
  Ground truth (1 of 5): A man opens pizza box and stares at the camera.

  Image      : 000000382030.jpg
  Predicted  : a cat laying on the floor next to a purse.
  Ground truth (1 of 5): A wig sitting next to a pair of shoes on top of a table.



In [32]:
import shutil, os

shutil.copy("/kaggle/working/blip_coco/best_model/model.safetensors",
            "/kaggle/working/model.safetensors")



print("✅ Now go to Output panel → click model.safetensors → Download")

✅ Now go to Output panel → click model.safetensors → Download


In [33]:
import shutil

shutil.make_archive(
    '/kaggle/working/blip_model',
    'zip',
    '/kaggle/working/blip_coco/best_model'
)

'/kaggle/working/blip_model.zip'

In [34]:
from IPython.display import FileLink

FileLink('/kaggle/working/blip_model.zip')

/kaggle/working/blip_model.zip